In [1]:
import os
import json
from pathlib import Path
base_path = ""
tasks = ["majority", "binmajority", "binmajorityinterleave", "count", "sort", "uniquebigramcopy", "uniquecopy", "uniquereverse"]
models = os.listdir(base_path)
configs = {}
metrics = {}
primitives = {}
code = {}
for task in tasks:
    configs[task] = {}
    metrics[task] = {}
    primitives[task] = {}
    code[task] = {}
failed_models = []
success_models = []
for model in models:
    task = model.split("=")[0]
    nonempty_exp = []
    for exp in os.listdir(Path(base_path) / model):
        out_dir = Path(base_path) / model / exp
        if os.listdir(out_dir):
            nonempty_exp.append(out_dir)
    if len(nonempty_exp) != 1:
        failed_models.append(model)
        continue
    success_models.append(model)
    nonempty_exp = nonempty_exp[0]
    with open(nonempty_exp / "program_code.txt", "r") as file:
        code[task][model] = file.read()
    with open(nonempty_exp / "metrics.json", "r") as file:
        metrics[task][model] = json.load(file)
    with open(nonempty_exp / "config_without_select_project_noops.json", "r") as file:
        configs[task][model] = json.load(file)
    with open(nonempty_exp / "primitives_config.json", "r") as file:
        primitives[task][model] = json.load(file)

In [ ]:
path_to_good_models = ""
with open(path_to_good_models, "r") as file:
    good_models = json.load(file)
good_models = good_models["chosen_exps_new_random_seeds"] | good_models["chosen_exps_new_models"] | good_models["chosen_exps_unique_copy"]

In [3]:
models_have_good_pruned = ["".join((v.replace("_", "")).split("-")[1:4]) for v in good_models.keys() if good_models[v]]
models_have_good_pruned = [m for m in models_have_good_pruned if (m in success_models) or (m in failed_models)]
models_dont_have_good_pruned = ["".join((v.replace("_", "")).split("-")[1:4]) for v in good_models.keys() if not good_models[v]]
models_dont_have_good_pruned = [m for m in models_dont_have_good_pruned if (m in success_models) or (m in failed_models)]

In [4]:
print(set(models_have_good_pruned) == set(success_models))
print(set(models_dont_have_good_pruned) == set(failed_models))

True
True


In [7]:
model_compressed_name_to_full = {
    "".join((v.replace("_", "")).split("-")[1:4]): "-".join(v.split("-")[1:4])
    for v in good_models.keys()
}

In [ ]:
from patching_data import get_tokenizer_and_dataset_for_task
from train_new_models import customCollator, customBCECollator
from transformers import GPT2LMHeadModel
import torch

PATH_TO_SAVED_MODELS = ""

def tes_len_gen(model_name, task_name):
    num_test_step = 5
    batch_size = 120
    orig_model = GPT2LMHeadModel.from_pretrained(f"/{PATH_TO_SAVED_MODELS}/{model_name}")
    orig_model.eval()

    tokenizer, iterable_dataset = get_tokenizer_and_dataset_for_task(task_name, (50, 100), 150, {"period_for_data":3})
    if not hasattr(iterable_dataset, "BCE") or not iterable_dataset.BCE:
        collator = customCollator(tokenizer.pad_token_id)
    else:
        collator = customBCECollator(tokenizer.pad_token_id)
    inputs = []
    num_correct_items = 0
    num_items = 0
    step = 0
    with torch.no_grad():
        for item in iterable_dataset:
            inputs.append(item)
            if len(inputs) == batch_size:
                batch = collator(inputs)
                batch = {k: v.to(orig_model.device) for k, v in batch.items()}
                labels = batch["labels"]
                batch.pop("labels")
                
                result = orig_model(**batch)
                logits = result.logits.detach()
                
                if not hasattr(iterable_dataset, "BCE") or not iterable_dataset.BCE:
                    shift_logits = logits[:, :-1].detach()
                    shift_labels = labels[:, 1:].detach()
                    predictions = shift_logits.argmax(dim=-1)

                    correct = ((predictions == shift_labels) | (shift_labels == -100)).all(dim=1)
                    num_correct_items += correct.sum().item()
                    num_items += len(correct)
                else:
                    predictions = (logits > 0).long()

                    mask = (batch["input_ids"] == tokenizer.pad_token_id) | (batch["input_ids"] == tokenizer.eos_token_id)
                    correct = ((predictions == labels).all(dim=-1) | mask).all(dim=1)
                    num_correct_items += correct.sum().item()
                    num_items += len(correct)
                step += 1
                inputs = []
                if step == num_test_step:
                    break
    return num_correct_items / num_items

In [ ]:
good_model_to_len_gen = {}
for model_name in models_have_good_pruned:
    acc = tes_len_gen(model_compressed_name_to_full[model_name], model_compressed_name_to_full[model_name].split("-")[0])
    good_model_to_len_gen[model_name] = acc

bad_model_to_len_gen = {}
for model_name in models_dont_have_good_pruned:
    acc = tes_len_gen(model_compressed_name_to_full[model_name], model_compressed_name_to_full[model_name].split("-")[0])
    bad_model_to_len_gen[model_name] = acc

In [57]:
from collections import defaultdict
lengen_per_task = defaultdict(list)
for model, lengen in (good_model_to_len_gen | bad_model_to_len_gen).items():
    task = model.split("=")[0]
    lengen_per_task[task].append(lengen)

In [ ]:
import re
from itertools import chain
from collections import defaultdict
from itertools import combinations

import torch

from patching_data import get_tokenizer_for_task
from primitives_for_coefficients import (
    ATTENTION_ALL_PRIMITIVES,
    ATTENTION_CONST_ALL_PRIMITIVES,
    LOGITS_ALL_PRIMITIVES,
    LOGITS_CONST_ALL_PRIMITIVES,
)


DEFAULT_MAX_TEST_LENGTH = 150

def pearson_corr(left, right):
    left = torch.as_tensor(left, dtype=torch.float32).flatten()
    right = torch.as_tensor(right, dtype=torch.float32).flatten()
    if left.shape != right.shape:
        return float("nan")

    left = left - left.mean()
    right = right - right.mean()
    denom = torch.linalg.vector_norm(left) * torch.linalg.vector_norm(right)
    if denom == 0:
        return float("nan")
    return (left @ right / denom).item()


def get_pairwise_primitive_corr_details_by_task(primitive_matrices, configs, model_to_task=None):
    models_by_task = defaultdict(list)
    for model_name in primitive_matrices:
        if model_to_task is not None:
            task_name = model_to_task[model_name]
        else:
            task_name = model_name.split("=")[0]
        models_by_task[task_name].append(model_name)
    assert len(models_by_task) == 1

    corr_details = []
    for task_name, model_names in models_by_task.items():
        for model_a, model_b in combinations(sorted(model_names), 2):
            primitives_a = primitive_matrices[model_a]
            primitives_b = primitive_matrices[model_b]
            if configs[model_b] != configs[model_a]:
                continue
            shared_endpoints = sorted(set(primitives_a) & set(primitives_b))

            endpoint_corrs = {}
            for endpoint in shared_endpoints:
                corr = pearson_corr(primitives_a[endpoint], primitives_b[endpoint])
                if torch.isfinite(torch.tensor(corr)):
                    endpoint_corrs[endpoint] = corr

            if endpoint_corrs:
                average_corr = sum(endpoint_corrs.values()) / len(endpoint_corrs)
            else:
                average_corr = float("nan")

            corr_details.append(
                {
                    "models": (model_a, model_b),
                    "average_corr": average_corr,
                    "endpoint_corrs": endpoint_corrs,
                }
            )

    return corr_details

def clean_program_text(program_text):
    cleaned_lines = []
    for line in program_text.splitlines():
        line = re.sub(r"\\circled\{[^}]*\}", "CIRCLED", line)
        cleaned_lines.append(line.split("#", 1)[0].strip())
    return re.sub(r"\s+", "", "\n".join(cleaned_lines))


def task_name_from_model_name(model_name):
    return model_name.split("-")[0]


def d_model_from_model_name(model_name):
    if model_name is None:
        return None
    match = re.search(r"(\d+)d(?:_|$)", model_name)
    return int(match.group(1)) if match else None


def infer_n_positions(task_name, max_test_length=DEFAULT_MAX_TEST_LENGTH):
    task_to_n_positions = {
        "bin_majority": max_test_length + 4,
        "majority": max_test_length + 4,
        "bin_majority_interleave": max_test_length + 6,
        "unique_copy": max_test_length * 2 + 3,
        "sort": max_test_length * 2 + 3,
        "unique_reverse": max_test_length * 2 + 3,
        "unique_bigram_copy": max_test_length * 2 + 3,
        "count": max_test_length + 5,
        "repeat_copy": max_test_length * 2 + 3,
        "addition": max_test_length * 2 + 2,
    }
    if task_name in task_to_n_positions:
        return task_to_n_positions[task_name]
    if task_name is not None and "tomita" in task_name:
        return max_test_length + 2
    return max_test_length + 4


def get_tokenizer(model_name=None, task_name=None, max_test_length=DEFAULT_MAX_TEST_LENGTH):
    if task_name is None:
        if model_name is None:
            raise ValueError("Pass either task_name or model_name so the tokenizer can be reconstructed.")
        task_name = task_name_from_model_name(model_name)
    return get_tokenizer_for_task(task_name, max_test_length)


def _normalise_primitives_config(config):
    return config["primitives"] if "primitives" in config else config


def _as_tensor(matrix_like):
    return torch.as_tensor(matrix_like, dtype=torch.float32)


def _saved_matrix_from_interaction(interaction):
    for key in ("replacement_matrix", "rounded_matrix", "primitive_matrix", "matrix", "tensor"):
        if key in interaction and interaction[key] is not None:
            return _as_tensor(interaction[key])
    return None


def _iter_interactions(config):
    config = _normalise_primitives_config(config)
    for layer, layer_config in config.items():
        if layer == "lm_head":
            continue
        for head, head_config in layer_config.items():
            yield from head_config.get("qk_interactions", [])
            yield from head_config.get("k_interactions", [])
    yield from config.get("lm_head", [])


def infer_activation_dims(config, tokenizer):
    dims = {
        "vocab_bias": len(tokenizer.vocab),
        "bias": len(tokenizer.vocab),
        "lm_head": len(tokenizer.vocab),
    }
    for interaction in _iter_interactions(config):
        matrix = _saved_matrix_from_interaction(interaction)
        if matrix is None:
            continue

        q_name = interaction.get("activation_name_to_keep_q")
        k_name = interaction.get("activation_name_to_keep_k")
        logits_name = interaction.get("activation_name_to_keep")

        if logits_name is not None:
            if matrix.ndim == 1:
                dims[logits_name] = matrix.shape[0]
            elif matrix.ndim == 2:
                dims[logits_name] = matrix.shape[0]
            continue

        if q_name is None:
            if matrix.ndim == 1:
                dims[k_name] = matrix.shape[0]
            elif matrix.ndim == 2:
                dims[k_name] = matrix.shape[-1]
        elif matrix.ndim == 2:
            dims[q_name] = matrix.shape[0]
            dims[k_name] = matrix.shape[1]

    return dims


def _activation_nodes(act_name):
    if act_name is None:
        return []
    pattern = r"attn_output-\d+-\d+|mlp-\d+|lm_head|wte|wpe"
    return re.findall(pattern, act_name)


def infer_num_dims(
    act_name,
    tokenizer,
    *,
    config=None,
    activation_dims=None,
    max_position_embeddings=DEFAULT_MAX_TEST_LENGTH,
    model_name=None,
    model_dim=None,
):
    if act_name in (None, "bias", "vocab_bias", "lm_head"):
        return len(tokenizer.vocab)

    if activation_dims is None:
        activation_dims = infer_activation_dims(config, tokenizer) if config is not None else {}
    if act_name in activation_dims:
        return int(activation_dims[act_name])

    nodes = _activation_nodes(act_name)
    if not nodes:
        raise ValueError(f"Cannot infer dimensions for activation {act_name!r}.")

    if any(node.startswith("mlp") for node in nodes):
        if model_dim is None:
            model_dim = d_model_from_model_name(model_name)
        if model_dim is not None:
            return int(model_dim)

    inner_node = nodes[-1]
    if inner_node == "wte":
        return len(tokenizer.vocab)
    if inner_node == "wpe":
        return int(max_position_embeddings)
    if inner_node == "lm_head":
        return len(tokenizer.vocab)

    raise ValueError(
        f"Cannot infer dimensions for activation {act_name!r}. "
        "Pass activation_dims={...} or include at least one saved matrix with this activation in the config."
    )


def _find_predefined_primitive(config_item, primitives_set):
    predefined_primitive_name = config_item["predefined_primitive"]
    matches = [primitive for primitive in primitives_set if primitive.name == predefined_primitive_name]
    if len(matches) != 1:
        available = ", ".join(primitive.name for primitive in primitives_set)
        raise ValueError(f"Unknown primitive {predefined_primitive_name!r}. Available: {available}")
    return matches[0]


def get_predefined_primitive_matrix(config_item, dims_left, dims_right, tokenizer, primitives_set):
    predefined_primitive = _find_predefined_primitive(config_item, primitives_set)
    if dims_left is None:
        matrix = predefined_primitive.contruct_matrix(dims_right, tokenizer)
    else:
        matrix = predefined_primitive.contruct_matrix(dims_left, dims_right, tokenizer)

    if "predefined_special_primitive" in config_item:
        if dims_left is None:
            raise ValueError("Special primitives are only valid for two-dimensional primitives.")
        predefined_special_primitive = _find_predefined_primitive(
            {"predefined_primitive": config_item["predefined_special_primitive"]},
            primitives_set,
        )
        special_matrix = predefined_special_primitive.contruct_matrix(dims_left, dims_right, tokenizer)
        special_tokens = [
            tokenizer.sep_token_id,
            tokenizer.bos_token_id,
            tokenizer.eos_token_id,
            tokenizer.pad_token_id,
        ]
        matrix[special_tokens, :] = special_matrix[special_tokens, :]

    return matrix * config_item.get("scaling_factor_primitive", 1.0)


def _primitive_set_for_interaction(q=None, k=None, a=None):
    if q is None and k is None:
        return LOGITS_CONST_ALL_PRIMITIVES if a in ("bias", "vocab_bias") else LOGITS_ALL_PRIMITIVES
    return ATTENTION_CONST_ALL_PRIMITIVES if q in (None, "bias") else ATTENTION_ALL_PRIMITIVES


def get_primitive(
    interaction,
    q=None,
    k=None,
    a=None,
    *,
    config=None,
    model_name=None,
    task_name=None,
    tokenizer=None,
    activation_dims=None,
    max_test_length=DEFAULT_MAX_TEST_LENGTH,
    max_position_embeddings=None,
    model_dim=None,
):
    saved_matrix = _saved_matrix_from_interaction(interaction)
    if saved_matrix is not None:
        return saved_matrix

    if not interaction.get("predefined_primitive"):
        return None

    if tokenizer is None:
        tokenizer = get_tokenizer(model_name=model_name, task_name=task_name, max_test_length=max_test_length)
    if activation_dims is None and config is not None:
        activation_dims = infer_activation_dims(config, tokenizer)
    if max_position_embeddings is None:
        if task_name is None and model_name is not None:
            task_name = task_name_from_model_name(model_name)
        max_position_embeddings = infer_n_positions(task_name, max_test_length)
    if model_dim is None:
        model_dim = d_model_from_model_name(model_name)

    if a is None and q is None and k is None:
        a = interaction.get("activation_name_to_keep")
    if k is None:
        k = interaction.get("activation_name_to_keep_k")
    if q is None and "activation_name_to_keep_q" in interaction:
        q = interaction.get("activation_name_to_keep_q")

    dims_left = None
    dims_right = None
    if q not in (None, "bias"):
        dims_left = infer_num_dims(
            q,
            tokenizer,
            config=config,
            activation_dims=activation_dims,
            max_position_embeddings=max_position_embeddings,
            model_name=model_name,
            model_dim=model_dim,
        )

    if k is not None:
        dims_right = infer_num_dims(
            k,
            tokenizer,
            config=config,
            activation_dims=activation_dims,
            max_position_embeddings=max_position_embeddings,
            model_name=model_name,
            model_dim=model_dim,
        )
    else:
        dims_right = len(tokenizer.vocab)
        if a not in (None, "bias", "vocab_bias"):
            dims_left = infer_num_dims(
                a,
                tokenizer,
                config=config,
                activation_dims=activation_dims,
                max_position_embeddings=max_position_embeddings,
                model_name=model_name,
                model_dim=model_dim,
            )

    primitives_set = _primitive_set_for_interaction(q=q, k=k, a=a)
    return get_predefined_primitive_matrix(interaction, dims_left, dims_right, tokenizer, primitives_set)


def get_primitive_name_to_primitive_from_config(
    config,
    *,
    model_name=None,
    task_name=None,
    tokenizer=None,
    max_test_length=DEFAULT_MAX_TEST_LENGTH,
    max_position_embeddings=None,
    activation_dims=None,
    model_dim=None,
):
    config = _normalise_primitives_config(config)
    if tokenizer is None:
        tokenizer = get_tokenizer(model_name=model_name, task_name=task_name, max_test_length=max_test_length)
    if activation_dims is None:
        activation_dims = infer_activation_dims(config, tokenizer)
    if max_position_embeddings is None:
        if task_name is None and model_name is not None:
            task_name = task_name_from_model_name(model_name)
        max_position_embeddings = infer_n_positions(task_name, max_test_length)
    if model_dim is None:
        model_dim = d_model_from_model_name(model_name)

    primitive_name_to_primitive = {}
    for layer, layer_config in config.items():
        if layer == "lm_head":
            continue
        for head, head_config in layer_config.items():
            interactions = chain(
                head_config.get("qk_interactions", []),
                head_config.get("k_interactions", []),
            )
            for interaction in interactions:
                q = interaction.get("activation_name_to_keep_q")
                k = interaction["activation_name_to_keep_k"]
                q_name = "bias" if q is None else q
                primitive_name = f"{layer}.{head}.{q_name}@{k}"
                primitive_name_to_primitive[primitive_name] = get_primitive(
                    interaction,
                    q=q,
                    k=k,
                    config=config,
                    tokenizer=tokenizer,
                    activation_dims=activation_dims,
                    max_test_length=max_test_length,
                    max_position_embeddings=max_position_embeddings,
                    model_name=model_name,
                    model_dim=model_dim,
                )

    for interaction in config.get("lm_head", []):
        a = interaction["activation_name_to_keep"]
        a_name = "bias" if a == "vocab_bias" else a
        primitive_name = f"lm_head.{a_name}"
        primitive_name_to_primitive[primitive_name] = get_primitive(
            interaction,
            a=a_name,
            config=config,
            tokenizer=tokenizer,
            activation_dims=activation_dims,
            max_test_length=max_test_length,
            max_position_embeddings=max_position_embeddings,
            model_name=model_name,
            model_dim=model_dim,
        )

    return primitive_name_to_primitive

In [ ]:
primitive_matrices = {
    task: {
        model: get_primitive_name_to_primitive_from_config(config, model_name=model)
        for model, config in primitives[task].items()
    } for task in primitives
}

In [27]:
primitive_corr_details = {}
for task in primitive_matrices:
    primitive_corr_details[task] = get_pairwise_primitive_corr_details_by_task(primitive_matrices[task], configs[task])

In [47]:
import pandas as pd


def results_to_dataframe(
    tasks,
    configs,
    lengen_per_task,
    code,
    primitive_corr_details,
    clean_program_text,
):
    rows = []

    for task in tasks:
        corr_details = primitive_corr_details[task]

        row = {
            "model": task,
            "decompiled": len(configs[task]),
            "length-leneralized": sum([t >= 0.9 for t in lengen_per_task[task]]),
            "unique configs": len(set(map(str, [v["config"] for v in primitives[task].values()]))),
            "unique programs": len(set(map(clean_program_text, code[task].values()))),
            "avg. primitives person corr": (
                f'{sum(endpoint["average_corr"] for endpoint in corr_details) / len(corr_details):.2f}'
                if len(corr_details) > 0
                else None
            ),
        }

        rows.append(row)

    return pd.DataFrame(rows).set_index("model")


In [58]:
df = results_to_dataframe(
    tasks,
    configs,
    lengen_per_task,
    code,
    primitive_corr_details,
    clean_program_text,
)

df

,decompiled,length-leneralized,unique configs,unique programs,avg. primitives person corr
model,,,,,
majority,10,10,7,1,nan
binmajority,10,10,6,1,1.00
binmajorityinterleave,2,2,2,2,None
count,1,10,1,1,None
sort,10,10,1,2,0.96
uniquebigramcopy,7,10,7,4,None
uniquecopy,7,6,3,3,0.94
uniquereverse,9,9,5,4,0.67
